# Notebook 06 — Adversarial Robustness Ensemble

**Project:** AV-XAI
**Goal:** Train 4 augmentation-specialized variants by fine-tuning from `best.pt`, then perform soft-vote ensemble inference and regenerate paper tables/figures with the ensemble.

## The five ensemble members

| # | Name | Augmentations | Audio aug? |
|---|---|---|---|
| M0 | Clean baseline (existing `best.pt`) | None | — |
| M1 | Heavy JPEG | Face JPEG q∈[20, 40] | No |
| M2 | Light JPEG | Face JPEG q∈[60, 95] (matches paper §6.2) | No |
| M3 | Gaussian noise | Face σ∈[0.01, 0.05] + audio SNR ∈[15, 25] dB | **Yes** |
| M4 | Mixed | Random combination of above + temporal frame resampling | **Yes** |

M3 and M4 include audio augmentation to address the paper's §3 motivation:
> "Adversarial robustness improvements operate primarily in the visual domain, neglecting acoustic robustness."

## Fine-tuning hyperparameters

Lower LR (half of nb04) to avoid catastrophic forgetting from `best.pt`:

| Parameter | Value |
|---|---|
| Backbone LR | 5e-6 |
| Head LR | 2.5e-4 |
| Epochs per variant | 3 |
| Warmup steps | 100 |
| Audio mixup | Disabled (each variant has its own augmentation story) |
| Other hyperparameters | Same as nb04 |

## Estimated runtime

- Training: 4 variants × 3 epochs × ~45 min = **~9 hours**
- Ensemble inference: 5 models × ~12 min = ~1 hour
- Tables/figures: ~5 min
- **Total: ~10 hours**

## What gets generated

```
checkpoints/
├── best.pt                          ← M0 (existing, untouched)
├── M1_heavy_jpeg.pt
├── M2_light_jpeg.pt
├── M3_gaussian.pt
└── M4_mixed.pt

results/
├── tables/
│   ├── ensemble_main_results.csv    ← T1 update — with ensemble
│   ├── ensemble_binary_results.csv  ← T2 update
│   ├── ensemble_member_comparison.csv  ← M0-M4 individual + ensemble
│   ├── ensemble_baseline_comparison.csv ← T6 update
│   └── ensemble_per_class_breakdown.csv ← T3 update
└── figures/
    ├── ensemble_member_bars.{png,pdf}
    ├── ensemble_confusion_matrix.{png,pdf}
    └── ensemble_roc.{png,pdf}
```

In [ ]:
# --- Imports and paths ---
import os, sys, json, time, math, random, csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, roc_curve, auc as sk_auc,
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

from tqdm.auto import tqdm

# --- Paths ---
PROJECT_ROOT = Path(r"C:\Users\Komal-Sch\Desktop\AV-XAI")
DATA_ROOT    = PROJECT_ROOT / "data"
PREPROC_ROOT = DATA_ROOT / "preprocessed"
SPLITS_DIR   = DATA_ROOT / "splits"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
RESULTS_DIR  = PROJECT_ROOT / "results"
TABLES_DIR   = RESULTS_DIR / "tables"
FIGURES_DIR  = RESULTS_DIR / "figures"
LOGITS_DIR   = CKPT_DIR / "logits"   # per-variant test logits cached here

for d in (TABLES_DIR, FIGURES_DIR, LOGITS_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
from models.avxai_model import AVXAIModel

# --- Reproducibility ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Training requires CUDA."
print(f"Device: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

# --- JPEG augmentation availability check ---
try:
    from torchvision.transforms.v2 import functional as TF_v2
    JPEG_OK = True
    print("torchvision.transforms.v2.functional.jpeg available — fast JPEG augmentation.")
except ImportError:
    JPEG_OK = False
    print("WARNING: torchvision.transforms.v2 not available — JPEG augmentation will be slow PIL-based.")
    from PIL import Image
    import io

# --- Class labels ---
LABEL_NAMES_4 = ["real", "visual_modified", "audio_modified", "both_modified"]

# --- Fine-tuning config ---
FT_CFG = {
    "lr_backbone":       5e-6,
    "lr_head":           2.5e-4,
    "weight_decay":      0.01,
    "warmup_steps":      100,
    "ft_epochs":         3,
    "micro_batch":       4,
    "grad_accum":        8,
    "grad_clip":         1.0,
    "label_smoothing":   0.1,
    "tau_init":          0.0765,    # carry over from nb04
    "contrastive_weight": 0.3,
    "aux_loss_weight":   0.3,
    "num_workers":       0,
}
print("\nFine-tuning config:")
for k, v in FT_CFG.items(): print(f"  {k:<20}: {v}")

## Phase A — Augmentation functions

Each function operates on a GPU batch and returns the augmented batch. Functions are batchwise — same random parameters applied to all clips in a micro-batch for efficiency.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# JPEG compression on face crops
# ─────────────────────────────────────────────────────────────────────────────
def apply_jpeg(faces_uint8, q_low, q_high):
    """Apply JPEG compression at random quality.

    faces: [B, T, 3, H, W] uint8 (on GPU)
    Returns same shape, same dtype.
    """
    q = random.randint(q_low, q_high)
    B, T, C, H, W = faces_uint8.shape
    flat = faces_uint8.reshape(B * T, C, H, W)

    if JPEG_OK:
        # GPU-fast path (torchvision v2)
        flat_cpu = flat.cpu()                       # JPEG runs on CPU
        compressed = TF_v2.jpeg(flat_cpu, q)
        return compressed.to(faces_uint8.device).reshape(B, T, C, H, W)
    else:
        # PIL fallback (slow)
        out = torch.empty_like(flat)
        for i in range(flat.size(0)):
            img = Image.fromarray(flat[i].permute(1,2,0).cpu().numpy())
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=q)
            buf.seek(0)
            jpeg_img = Image.open(buf).convert("RGB")
            out[i] = torch.from_numpy(np.array(jpeg_img)).permute(2,0,1)
        return out.to(faces_uint8.device).reshape(B, T, C, H, W)


# ─────────────────────────────────────────────────────────────────────────────
# Gaussian noise on face crops (in normalized [0,1] space, then back to uint8)
# ─────────────────────────────────────────────────────────────────────────────
def apply_gaussian_face(faces_uint8, sigma_low=0.01, sigma_high=0.05):
    sigma = random.uniform(sigma_low, sigma_high)
    f = faces_uint8.float() / 255.0
    noise = torch.randn_like(f) * sigma
    f = (f + noise).clamp(0, 1)
    return (f * 255).to(torch.uint8)


# ─────────────────────────────────────────────────────────────────────────────
# Audio SNR-based noise injection
# ─────────────────────────────────────────────────────────────────────────────
def apply_audio_snr(audio, snr_db_low=15.0, snr_db_high=25.0):
    """Add white Gaussian noise calibrated to a target SNR per sample."""
    snr_db = random.uniform(snr_db_low, snr_db_high)
    snr_linear = 10 ** (snr_db / 10)
    sig_power = audio.pow(2).mean(dim=-1, keepdim=True).clamp(min=1e-10)
    noise_power = sig_power / snr_linear
    noise = torch.randn_like(audio) * noise_power.sqrt()
    return audio + noise


# ─────────────────────────────────────────────────────────────────────────────
# Temporal frame resampling (drop+duplicate frames in random pattern)
# ─────────────────────────────────────────────────────────────────────────────
def apply_temporal_resample(faces_uint8):
    """Simulate frame drops/duplications by sampling with replacement."""
    T = faces_uint8.size(1)
    indices = torch.sort(torch.randint(0, T, (T,))).values
    return faces_uint8[:, indices]


# ─────────────────────────────────────────────────────────────────────────────
# Variant dispatcher
# ─────────────────────────────────────────────────────────────────────────────
def apply_variant_augmentation(faces, audio, variant_cfg):
    """Apply the augmentation policy for one variant.

    variant_cfg: dict describing which augmentations to apply.
    Returns (faces_aug, audio_aug).
    """
    aug = variant_cfg["augmentations"]

    if "jpeg_quality" in aug:
        q_low, q_high = aug["jpeg_quality"]
        # Per-batch probability of applying
        if random.random() < aug.get("jpeg_prob", 1.0):
            faces = apply_jpeg(faces, q_low, q_high)

    if "gaussian_sigma" in aug:
        s_low, s_high = aug["gaussian_sigma"]
        if random.random() < aug.get("gaussian_prob", 1.0):
            faces = apply_gaussian_face(faces, s_low, s_high)

    if "audio_snr_db" in aug:
        snr_low, snr_high = aug["audio_snr_db"]
        if random.random() < aug.get("audio_snr_prob", 1.0):
            audio = apply_audio_snr(audio, snr_low, snr_high)

    if aug.get("temporal_resample", False):
        if random.random() < aug.get("temporal_resample_prob", 1.0):
            faces = apply_temporal_resample(faces)

    return faces, audio


print("Augmentation functions defined.")

## Phase B — Variant configurations

Each dict defines one ensemble member: name, save path, and augmentation policy.

In [ ]:
VARIANT_CONFIGS = [
    {
        "name":      "M1_heavy_jpeg",
        "ckpt":      CKPT_DIR / "M1_heavy_jpeg.pt",
        "augmentations": {
            "jpeg_quality": (20, 40),
            "jpeg_prob":    1.0,           # always apply on this variant
        },
    },
    {
        "name":      "M2_light_jpeg",
        "ckpt":      CKPT_DIR / "M2_light_jpeg.pt",
        "augmentations": {
            "jpeg_quality": (60, 95),      # matches paper §6.2
            "jpeg_prob":    1.0,
        },
    },
    {
        "name":      "M3_gaussian",
        "ckpt":      CKPT_DIR / "M3_gaussian.pt",
        "augmentations": {
            "gaussian_sigma": (0.01, 0.05),   # matches paper §6.2
            "gaussian_prob":  1.0,
            "audio_snr_db":   (15.0, 25.0),
            "audio_snr_prob": 1.0,
        },
    },
    {
        "name":      "M4_mixed",
        "ckpt":      CKPT_DIR / "M4_mixed.pt",
        "augmentations": {
            # Stochastic mix — any subset applied per batch
            "jpeg_quality":    (40, 80),
            "jpeg_prob":       0.5,
            "gaussian_sigma":  (0.01, 0.04),
            "gaussian_prob":   0.5,
            "audio_snr_db":    (18.0, 30.0),
            "audio_snr_prob":  0.5,
            "temporal_resample": True,
            "temporal_resample_prob": 0.5,
        },
    },
]

print(f"Defined {len(VARIANT_CONFIGS)} variants:")
for v in VARIANT_CONFIGS:
    print(f"  {v['name']:<18}  augs: {list(v['augmentations'].keys())}")

## Phase C — Fine-tuning loop (shared)

`finetune_variant` is called once per variant. It:
1. Loads `best.pt` weights into a fresh model (NOT the previous variant — every fine-tune starts from M0)
2. Builds a fresh optimizer + cosine schedule over `ft_epochs`
3. Trains for 3 epochs with the variant's augmentation policy applied per micro-batch
4. Saves the variant's checkpoint

In [ ]:
# --- Dataset (reuse the one from nb04 logic, inlined here for self-containment) ---
class AVXAIDataset(Dataset):
    def __init__(self, manifest_csv, preproc_root):
        df = pd.read_csv(manifest_csv)
        self.paths = [Path(preproc_root) / row["split"] / f"{Path(row['file']).stem}.pt"
                      for _, row in df.iterrows()]
        self.labels_4c  = df["label_4class"].astype(int).values
        self.labels_bin = df["label_binary"].astype(int).values

    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        e = torch.load(self.paths[idx], weights_only=False)
        return {"faces": e["faces"], "audio": e["audio"],
                "label_4c": int(self.labels_4c[idx]),
                "label_bin": int(self.labels_bin[idx])}

def collate(batch):
    return {
        "faces":    torch.stack([b["faces"] for b in batch], dim=0),
        "audio":    torch.stack([b["audio"] for b in batch], dim=0),
        "label_4c": torch.tensor([b["label_4c"] for b in batch], dtype=torch.long),
    }

train_ds = AVXAIDataset(SPLITS_DIR / "train.csv", PREPROC_ROOT)
print(f"Train set: {len(train_ds):,} clips")

# --- Loss function (carried from nb04, same structure) ---
def compute_loss(out, labels_4c, log_tau, cfg):
    ls = cfg["label_smoothing"]
    l_main = F.nll_loss(out["logits_4class"], labels_4c)
    l_v = F.cross_entropy(out["logits_video"], labels_4c, label_smoothing=ls)
    l_a = F.cross_entropy(out["logits_audio"], labels_4c, label_smoothing=ls)
    l_f = F.cross_entropy(out["logits_fused"], labels_4c, label_smoothing=ls)
    l_det = l_main + cfg["aux_loss_weight"] * (l_v + l_a + l_f) / 3.0

    tau = log_tau.exp()
    v_emb, a_emb = out["v_embed"], out["a_embed"]
    B = v_emb.size(0)
    sim = v_emb @ a_emb.T / tau
    targets = torch.arange(B, device=sim.device)
    l_cons = 0.5 * (F.cross_entropy(sim, targets) + F.cross_entropy(sim.T, targets))

    return {"total": l_det + cfg["contrastive_weight"] * l_cons,
            "det":   l_det.detach(),
            "cons":  l_cons.detach()}

# --- Cosine schedule with warmup ---
def build_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# --- Optimizer with 4 param groups ---
def is_no_decay(name):
    return name.endswith(".bias") or "layernorm" in name.lower() or "layer_norm" in name.lower() or "norm.weight" in name.lower()

def build_optimizer(model, log_tau, cfg):
    bb_d, bb_nd, hd_d, hd_nd = [], [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        is_bb = name.startswith("vivit") or name.startswith("wavlm")
        no_decay = is_no_decay(name)
        if is_bb:  (bb_nd if no_decay else bb_d).append(p)
        else:      (hd_nd if no_decay else hd_d).append(p)
    return torch.optim.AdamW([
        {"params": bb_d,  "lr": cfg["lr_backbone"], "weight_decay": cfg["weight_decay"]},
        {"params": bb_nd, "lr": cfg["lr_backbone"], "weight_decay": 0.0},
        {"params": hd_d,  "lr": cfg["lr_head"],     "weight_decay": cfg["weight_decay"]},
        {"params": hd_nd, "lr": cfg["lr_head"],     "weight_decay": 0.0},
        {"params": [log_tau], "lr": cfg["lr_head"], "weight_decay": 0.0},
    ])


# --- Fine-tune one variant ---
def finetune_variant(variant_cfg, base_ckpt_path, train_ds, cfg):
    """Train one ensemble variant from base checkpoint with its augmentation policy."""
    print(f"\n{'='*70}\nFINE-TUNING {variant_cfg['name']}\n{'='*70}")
    if variant_cfg["ckpt"].exists():
        print(f"Checkpoint already exists at {variant_cfg['ckpt']}. Skipping.")
        return

    # --- Build fresh model + load M0 weights ---
    torch.cuda.empty_cache()
    model = AVXAIModel(use_gradient_checkpointing=True).to(device)
    base_ck = torch.load(base_ckpt_path, weights_only=False, map_location=device)
    model.load_state_dict(base_ck["model_state"])
    log_tau = nn.Parameter(torch.log(torch.tensor(cfg["tau_init"])).to(device))

    optimizer = build_optimizer(model, log_tau, cfg)

    steps_per_epoch = math.ceil(len(train_ds) / (cfg["micro_batch"] * cfg["grad_accum"]))
    total_steps = steps_per_epoch * cfg["ft_epochs"]
    scheduler = build_scheduler(optimizer, total_steps, cfg["warmup_steps"])

    train_loader = DataLoader(train_ds, batch_size=cfg["micro_batch"], shuffle=True,
                              num_workers=cfg["num_workers"], collate_fn=collate,
                              pin_memory=True)

    print(f"Starting fine-tuning: {cfg['ft_epochs']} epochs, {total_steps} steps")
    print(f"  augmentations: {variant_cfg['augmentations']}")

    epoch_history = []
    t_run_start = time.time()

    for epoch in range(cfg["ft_epochs"]):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        accum = 0
        losses_epoch = defaultdict(list)
        n_batches = len(train_loader)
        log_every = max(1, n_batches // 30)
        t_epoch = time.time()

        for batch_idx, batch in enumerate(train_loader):
            faces = batch["faces"].to(device, non_blocking=True)
            audio = batch["audio"].to(device, non_blocking=True)
            labels = batch["label_4c"].to(device, non_blocking=True)

            # APPLY VARIANT AUGMENTATION
            faces, audio = apply_variant_augmentation(faces, audio, variant_cfg)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model(faces, audio)
                losses = compute_loss(out, labels, log_tau, cfg)
                loss_scaled = losses["total"] / cfg["grad_accum"]

            loss_scaled.backward()
            accum += 1

            if accum == cfg["grad_accum"]:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad] + [log_tau],
                    max_norm=cfg["grad_clip"],
                )
                optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                accum = 0

            for k in ("total", "det", "cons"):
                losses_epoch[k].append(float(losses[k]))

            if (batch_idx + 1) % log_every == 0 or batch_idx == n_batches - 1:
                pct = 100 * (batch_idx + 1) / n_batches
                elapsed_e = (time.time() - t_epoch) / 60
                speed = (batch_idx + 1) / max(0.001, time.time() - t_epoch)
                eta_e = (n_batches - batch_idx - 1) / max(0.001, speed) / 60
                print(f"  [{variant_cfg['name']}] E{epoch+1}/{cfg['ft_epochs']} "
                      f"[{batch_idx+1:>4}/{n_batches}] {pct:5.1f}%  "
                      f"loss={losses['total']:.3f} det={losses['det']:.3f} cons={losses['cons']:.3f}  "
                      f"elapsed={elapsed_e:.1f}m ETA={eta_e:.1f}m", flush=True)

        epoch_summary = {k: float(np.mean(v)) for k, v in losses_epoch.items()}
        epoch_summary["epoch"] = epoch + 1
        epoch_summary["epoch_time_min"] = (time.time() - t_epoch) / 60
        epoch_history.append(epoch_summary)
        print(f"  [{variant_cfg['name']}] Epoch {epoch+1} done: "
              f"total={epoch_summary['total']:.4f} "
              f"det={epoch_summary['det']:.4f} cons={epoch_summary['cons']:.4f} "
              f"time={epoch_summary['epoch_time_min']:.1f}m")

    # Save checkpoint
    save_payload = {
        "model_state":    model.state_dict(),
        "variant_name":   variant_cfg["name"],
        "augmentations":  variant_cfg["augmentations"],
        "epoch_history":  epoch_history,
        "ft_cfg":         cfg,
        "base_ckpt":      str(base_ckpt_path),
    }
    torch.save(save_payload, variant_cfg["ckpt"])

    elapsed = (time.time() - t_run_start) / 60
    print(f"\n  [{variant_cfg['name']}] Saved checkpoint. Total time: {elapsed:.1f} min")

    # Free
    del model, optimizer, scheduler, log_tau, train_loader
    torch.cuda.empty_cache()

print("finetune_variant() defined.")

## Phase D — Train all 4 variants

Sequential training. Each variant takes ~2.25 hours. **Total ~9 hours.** Same disconnect-safety as nb04 — disable Windows sleep, don't close Jupyter. Each variant's checkpoint is saved immediately so a crash mid-training only loses that variant.

If any variant fails, you can re-run this cell — already-completed variants are skipped (we check `if variant_cfg['ckpt'].exists()`).

In [ ]:
BASE_CKPT = CKPT_DIR / "best.pt"
assert BASE_CKPT.exists(), f"M0 base checkpoint not found at {BASE_CKPT}. Run nb04 first."

run_start = time.time()
for variant_cfg in VARIANT_CONFIGS:
    finetune_variant(variant_cfg, BASE_CKPT, train_ds, FT_CFG)

total_hours = (time.time() - run_start) / 3600
print(f"\n{'='*70}\nAll variants trained. Total elapsed: {total_hours:.2f} hours\n{'='*70}")
for v in VARIANT_CONFIGS:
    exists = v["ckpt"].exists()
    print(f"  {v['name']:<18}: {'OK' if exists else 'MISSING'}  {v['ckpt']}")

## Phase E — Per-variant test-set inference

For each model (M0…M4), run inference on the full test set, save log-softmax probabilities to `checkpoints/logits/{name}.npy`. After collecting all five, the ensemble combines them via simple soft vote (average of softmax probabilities).

To save VRAM, we load one model at a time and free it before the next.

In [ ]:
# --- Eval dataset ---
class AVXAIEvalDataset(Dataset):
    def __init__(self, manifest_csv, preproc_root):
        df = pd.read_csv(manifest_csv)
        self.paths = [Path(preproc_root) / row["split"] / f"{Path(row['file']).stem}.pt"
                      for _, row in df.iterrows()]
        self.labels_4c  = df["label_4class"].astype(int).values
        self.labels_bin = df["label_binary"].astype(int).values

    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        e = torch.load(self.paths[idx], weights_only=False)
        return {"faces": e["faces"], "audio": e["audio"],
                "label_4c": int(self.labels_4c[idx]),
                "label_bin": int(self.labels_bin[idx])}

def eval_collate(batch):
    return {
        "faces":    torch.stack([b["faces"] for b in batch], dim=0),
        "audio":    torch.stack([b["audio"] for b in batch], dim=0),
        "label_4c": torch.tensor([b["label_4c"] for b in batch], dtype=torch.long),
        "label_bin":torch.tensor([b["label_bin"] for b in batch], dtype=torch.long),
    }

test_ds = AVXAIEvalDataset(SPLITS_DIR / "test.csv", PREPROC_ROOT)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False,
                         num_workers=0, collate_fn=eval_collate, pin_memory=True)
print(f"Test set: {len(test_ds):,} clips")

# --- Inference for one model ---
def infer_model(ckpt_path, variant_name):
    """Load model, infer on test set, return (labels_4c, labels_bin, probs_4c)."""
    logits_fp = LOGITS_DIR / f"{variant_name}.npy"
    if logits_fp.exists():
        print(f"  Cached logits found for {variant_name}, loading...")
        cached = np.load(logits_fp, allow_pickle=True).item()
        return cached["labels_4c"], cached["labels_bin"], cached["probs_4c"]

    print(f"  Loading {ckpt_path.name}...")
    torch.cuda.empty_cache()
    model = AVXAIModel(use_gradient_checkpointing=False).to(device)
    ck = torch.load(ckpt_path, weights_only=False, map_location=device)
    model.load_state_dict(ck["model_state"])
    model.eval()

    all_logits, all_lab4, all_labbin = [], [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Inference [{variant_name}]"):
            faces = batch["faces"].to(device, non_blocking=True)
            audio = batch["audio"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = model(faces, audio)
            all_logits.append(out["logits_4class"].float().cpu())   # log-softmax
            all_lab4.append(batch["label_4c"])
            all_labbin.append(batch["label_bin"])

    logits = torch.cat(all_logits, dim=0).numpy()
    labels_4c  = torch.cat(all_lab4, dim=0).numpy()
    labels_bin = torch.cat(all_labbin, dim=0).numpy()
    probs_4c = np.exp(logits)   # log-softmax → probs

    # Cache
    np.save(logits_fp, {"labels_4c": labels_4c, "labels_bin": labels_bin, "probs_4c": probs_4c})
    print(f"  Saved logits cache: {logits_fp}")

    del model
    torch.cuda.empty_cache()
    return labels_4c, labels_bin, probs_4c


# --- Run inference for all 5 models ---
all_model_probs = {}
labels_4c = labels_bin = None

# M0 (base)
labels_4c, labels_bin, probs = infer_model(BASE_CKPT, "M0_base")
all_model_probs["M0_base"] = probs

# M1–M4
for variant_cfg in VARIANT_CONFIGS:
    _, _, probs = infer_model(variant_cfg["ckpt"], variant_cfg["name"])
    all_model_probs[variant_cfg["name"]] = probs

print(f"\nCollected probs from {len(all_model_probs)} models. Test set size: {len(labels_4c):,}")

## Phase F — Ensemble metrics + per-member comparison

The ensemble prediction is `mean(probs_M0, probs_M1, ..., probs_M4)`. Each member is also evaluated individually so we can show the contribution of each.

In [ ]:
# --- Compute per-model + ensemble metrics ---
def metrics_4class(probs, labels):
    preds = probs.argmax(axis=1)
    return {
        "Accuracy":      accuracy_score(labels, preds),
        "BalancedAcc":   balanced_accuracy_score(labels, preds),
        "AUC_macro_OvR": roc_auc_score(labels, probs, multi_class="ovr", average="macro"),
        "AUC_weighted":  roc_auc_score(labels, probs, multi_class="ovr", average="weighted"),
        "F1_macro":      f1_score(labels, preds, average="macro"),
        "F1_weighted":   f1_score(labels, preds, average="weighted"),
        "Precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "Recall_macro":  recall_score(labels, preds, average="macro"),
    }

# Per-member rows
member_rows = []
for name, probs in all_model_probs.items():
    m = {"Method": name, **metrics_4class(probs, labels_4c)}
    member_rows.append(m)

# Ensemble (soft vote)
ensemble_probs = np.mean(list(all_model_probs.values()), axis=0)
m_ens = {"Method": "AV-XAI Ensemble (M0-M4)", **metrics_4class(ensemble_probs, labels_4c)}
member_rows.append(m_ens)

member_df = pd.DataFrame(member_rows)
member_df.to_csv(TABLES_DIR / "ensemble_member_comparison.csv", index=False, float_format="%.4f")
print("=" * 80)
print("Ensemble member comparison")
print("=" * 80)
print(member_df.to_string(index=False, float_format="%.4f"))

# Highlight ensemble
ens_bacc = m_ens["BalancedAcc"]; ens_auc = m_ens["AUC_macro_OvR"]
m0_bacc = member_rows[0]["BalancedAcc"]
delta = ens_bacc - m0_bacc
print(f"\nEnsemble vs M0 base:")
print(f"  M0 base BAcc:        {m0_bacc:.4f}")
print(f"  Ensemble BAcc:       {ens_bacc:.4f}")
print(f"  Delta:               {delta:+.4f}  ({delta*100:+.2f} pp)")

## Phase G — Updated tables and figures

**Tables:**
- `ensemble_main_results.csv` — ensemble row + M0 baseline row for direct comparison
- `ensemble_binary_results.csv` — binary metrics from the ensemble's 4-class predictions
- `ensemble_per_class_breakdown.csv` — precision/recall/F1 per class for the ensemble
- `ensemble_baseline_comparison.csv` — Ours-Ensemble vs BA-TFD vs BA-TFD+

**Figures:**
- `ensemble_member_bars.png` — bar chart of all 5 members + ensemble
- `ensemble_confusion_matrix.png` — 4×4 from ensemble predictions
- `ensemble_roc.png` — ROC per class from ensemble

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tables
# ─────────────────────────────────────────────────────────────────────────────
ensemble_preds_4c = ensemble_probs.argmax(axis=1)
ensemble_preds_bin = (ensemble_preds_4c > 0).astype(int)
ensemble_probs_fake = 1.0 - ensemble_probs[:, 0]

# T1' main results (ensemble + base for comparison)
main_df = pd.DataFrame([
    {"Method": "AV-XAI M0 base only", **metrics_4class(all_model_probs["M0_base"], labels_4c)},
    {"Method": "AV-XAI Ensemble",     **metrics_4class(ensemble_probs, labels_4c)},
])
main_df.to_csv(TABLES_DIR / "ensemble_main_results.csv", index=False, float_format="%.4f")
print("T1': ensemble_main_results.csv")
print(main_df.to_string(index=False, float_format="%.4f"))

# T2' binary
binary_df = pd.DataFrame([
    {"Method":      "AV-XAI Ensemble (4-class derived)",
     "Accuracy":    accuracy_score(labels_bin, ensemble_preds_bin),
     "BalancedAcc": balanced_accuracy_score(labels_bin, ensemble_preds_bin),
     "AUC":         roc_auc_score(labels_bin, ensemble_probs_fake),
     "F1":          f1_score(labels_bin, ensemble_preds_bin),
     "Precision":   precision_score(labels_bin, ensemble_preds_bin, zero_division=0),
     "Recall":      recall_score(labels_bin, ensemble_preds_bin)},
])
binary_df.to_csv(TABLES_DIR / "ensemble_binary_results.csv", index=False, float_format="%.4f")
print("\nT2': ensemble_binary_results.csv")
print(binary_df.to_string(index=False, float_format="%.4f"))

# T3' per-class breakdown for ensemble
rep = classification_report(labels_4c, ensemble_preds_4c, target_names=LABEL_NAMES_4,
                            output_dict=True, zero_division=0)
per_class_df = pd.DataFrame([
    {"Class": cn,
     "Precision": rep[cn]["precision"],
     "Recall":    rep[cn]["recall"],
     "F1":        rep[cn]["f1-score"],
     "Support":   int(rep[cn]["support"])}
    for cn in LABEL_NAMES_4
])
per_class_df.to_csv(TABLES_DIR / "ensemble_per_class_breakdown.csv", index=False, float_format="%.4f")
print("\nT3': ensemble_per_class_breakdown.csv")
print(per_class_df.to_string(index=False, float_format="%.4f"))

# T6' baseline comparison (ensemble version)
ens_main = main_df[main_df["Method"] == "AV-XAI Ensemble"].iloc[0]
baselines_df = pd.DataFrame([
    {"Method":"BA-TFD",         "BAcc_4c":0.821, "AUC_4c":0.901, "BAcc_bin":0.875, "Note":"verify against paper"},
    {"Method":"BA-TFD+",        "BAcc_4c":0.853, "AUC_4c":0.927, "BAcc_bin":0.893, "Note":"verify against paper"},
    {"Method":"AV-XAI M0 only", "BAcc_4c":float(member_rows[0]["BalancedAcc"]),
     "AUC_4c":float(member_rows[0]["AUC_macro_OvR"]),
     "BAcc_bin":balanced_accuracy_score(labels_bin, (all_model_probs["M0_base"].argmax(1)>0).astype(int)),
     "Note":"single model"},
    {"Method":"AV-XAI Ensemble","BAcc_4c":float(ens_main["BalancedAcc"]),
     "AUC_4c":float(ens_main["AUC_macro_OvR"]),
     "BAcc_bin":float(binary_df["BalancedAcc"].iloc[0]),
     "Note":"5-model ensemble"},
])
baselines_df.to_csv(TABLES_DIR / "ensemble_baseline_comparison.csv", index=False, float_format="%.4f")
print("\nT6': ensemble_baseline_comparison.csv")
print(baselines_df.to_string(index=False, float_format="%.4f"))


# ─────────────────────────────────────────────────────────────────────────────
# Figures
# ─────────────────────────────────────────────────────────────────────────────
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
    print(f"  saved: {name}.png + {name}.pdf")

# F: ensemble member bars
fig, ax = plt.subplots(figsize=(10, 5))
names = [r["Method"] for r in member_rows]
baccs = [r["BalancedAcc"] for r in member_rows]
aucs  = [r["AUC_macro_OvR"] for r in member_rows]
f1s   = [r["F1_macro"] for r in member_rows]
x = np.arange(len(names))
width = 0.27

ax.bar(x - width, baccs, width, label="BAcc")
ax.bar(x,         aucs,  width, label="AUC (macro OvR)")
ax.bar(x + width, f1s,   width, label="F1 (macro)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("Score"); ax.set_ylim(0, 1)
ax.set_title("Ensemble member comparison")
ax.legend(); ax.grid(alpha=0.3, axis="y")

# Highlight the ensemble bar group
for i, (b, a, f) in enumerate(zip(baccs, aucs, f1s)):
    ax.text(i - width, b + 0.01, f"{b:.3f}", ha="center", fontsize=7)
    ax.text(i,         a + 0.01, f"{a:.3f}", ha="center", fontsize=7)
    ax.text(i + width, f + 0.01, f"{f:.3f}", ha="center", fontsize=7)

plt.tight_layout()
save_fig(fig, "ensemble_member_bars")
plt.show()

# F: ensemble confusion matrix
cm = confusion_matrix(labels_4c, ensemble_preds_4c)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(LABEL_NAMES_4, rotation=30, ha="right")
ax.set_yticklabels(LABEL_NAMES_4)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Ensemble confusion matrix (BAcc={ens_main['BalancedAcc']:.4f})")
for i in range(4):
    for j in range(4):
        v = cm_norm[i, j]
        c = "white" if v > 0.5 else "black"
        ax.text(j, i, f"{v:.2f}\n({cm[i,j]:,})", ha="center", va="center", color=c, fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_fig(fig, "ensemble_confusion_matrix")
plt.show()

# F: ensemble ROC curves
fig, ax = plt.subplots(figsize=(6.5, 5.5))
fpr, tpr, roc_auc = {}, {}, {}
for i, cn in enumerate(LABEL_NAMES_4):
    y_true_i = (labels_4c == i).astype(int)
    y_score_i = ensemble_probs[:, i]
    fpr[i], tpr[i], _ = roc_curve(y_true_i, y_score_i)
    roc_auc[i] = sk_auc(fpr[i], tpr[i])
    ax.plot(fpr[i], tpr[i], lw=2, label=f"{cn} (AUC={roc_auc[i]:.3f})")

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(4)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(4):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= 4
macro_auc = sk_auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "k--", lw=2, label=f"macro avg (AUC={macro_auc:.3f})")
ax.plot([0, 1], [0, 1], "gray", lw=1, alpha=0.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Ensemble ROC — 4-class OvR")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, "ensemble_roc")
plt.show()

## Phase H — Final summary

In [ ]:
ensemble_summary = {
    "method": "AV-XAI 5-model adversarial ensemble",
    "members": [r["Method"] for r in member_rows],
    "ensemble_metrics_4class": {
        k: float(member_rows[-1][k])
        for k in ("Accuracy", "BalancedAcc", "AUC_macro_OvR", "F1_macro")
    },
    "ensemble_metrics_binary": {
        "Accuracy":    float(binary_df["Accuracy"].iloc[0]),
        "BalancedAcc": float(binary_df["BalancedAcc"].iloc[0]),
        "AUC":         float(binary_df["AUC"].iloc[0]),
        "F1":          float(binary_df["F1"].iloc[0]),
    },
    "per_member_bacc": {r["Method"]: float(r["BalancedAcc"]) for r in member_rows},
    "delta_bacc_vs_M0": float(member_rows[-1]["BalancedAcc"] - member_rows[0]["BalancedAcc"]),
}

with open(RESULTS_DIR / "ensemble_metrics.json", "w") as f:
    json.dump(ensemble_summary, f, indent=2)

print("=" * 60)
print("ENSEMBLE HEADLINE NUMBERS")
print("=" * 60)
print(f"4-class:")
print(f"  Balanced accuracy : {ensemble_summary['ensemble_metrics_4class']['BalancedAcc']:.4f}")
print(f"  AUC (macro OvR)   : {ensemble_summary['ensemble_metrics_4class']['AUC_macro_OvR']:.4f}")
print(f"  F1 (macro)        : {ensemble_summary['ensemble_metrics_4class']['F1_macro']:.4f}")
print()
print(f"Binary (derived):")
print(f"  Balanced accuracy : {ensemble_summary['ensemble_metrics_binary']['BalancedAcc']:.4f}")
print(f"  AUC               : {ensemble_summary['ensemble_metrics_binary']['AUC']:.4f}")
print()
print(f"Per-member BAcc:")
for k, v in ensemble_summary["per_member_bacc"].items():
    print(f"  {k:<30}: {v:.4f}")
print()
print(f"Delta vs M0 base: {ensemble_summary['delta_bacc_vs_M0']:+.4f} ({ensemble_summary['delta_bacc_vs_M0']*100:+.2f} pp)")
print(f"\nSaved: {RESULTS_DIR / 'ensemble_metrics.json'}")

## Done.

**Git commit:**
```cmd
git add notebooks/06_adversarial_ensemble.ipynb results/tables/ensemble_*.csv results/figures/ensemble_*.png results/figures/ensemble_*.pdf results/ensemble_metrics.json
git add -f checkpoints/M1_heavy_jpeg.pt   # only if you want to commit checkpoints (~1.6 GB each — likely don't)
git commit -m "nb06: 5-model adversarial ensemble — BAcc=X.XXXX, delta=+Y.YY pp vs M0"
git push origin main
```

**Don't commit the variant checkpoints** — they're 1.6 GB each. Treat them like `best.pt`: kept locally, regenerated from scratch if needed.

**Next: nb07 — XAI visualization.**
- Grad-CAM++ on ViViT (spatial saliency on face crops)
- SHAP on WavLM (audio time-feature attribution)
- LRP on fusion module (cross-modal relevance)
- F6/F7 figures for the paper